In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 - Bronze Layer: Raw Ingestion
# MAGIC
# MAGIC **Goal:** land the raw source files into Delta with zero transformation, so we always
# MAGIC have an unmodified copy of what the client sent us. This is our replay / audit source
# MAGIC of truth.
# MAGIC
# MAGIC Techniques demonstrated:
# MAGIC - **Auto Loader** (`cloudFiles`) for incremental, exactly-once file discovery
# MAGIC - Separate streams per source **format** (CSV and JSON), unioned into one Bronze table
# MAGIC - **Schema inference + evolution** (`addNewColumns`) with a persisted schema location
# MAGIC - **Checkpointing** so re-running the job never reprocesses already-ingested files
# MAGIC - `trigger(availableNow=True)` - runs like a batch job (process what's there, then stop),
# MAGIC   which suits a scheduled Databricks Job while still using the streaming engine's
# MAGIC   exactly-once file tracking
# MAGIC - Ingestion metadata columns for lineage/debugging (`_source_file`, `_ingest_ts`, `_rescued_data`)

# COMMAND ----------

from pyspark.sql import functions as F

CATALOG = "lakehouse_demo"
SCHEMA = "transactions"

LANDING_CSV = f"/Volumes/{CATALOG}/{SCHEMA}/landing/csv"
LANDING_JSON = f"/Volumes/{CATALOG}/{SCHEMA}/landing/json"
SCHEMA_LOC = f"/Volumes/{CATALOG}/{SCHEMA}/schemas"
CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_transactions"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Helper: run one Auto Loader stream to completion (batch-style, idempotent)

# COMMAND ----------

def ingest_source(source_format: str, source_path: str):
    """
    Reads new files from `source_path` using Auto Loader and appends them to the Bronze
    Delta table. Safe to re-run: Auto Loader's checkpoint tracks exactly which files have
    already been processed, so already-seen files are skipped automatically.
    """
    checkpoint_path = f"{CHECKPOINT_BASE}/bronze_{source_format}"
    schema_location = f"{SCHEMA_LOC}/bronze_{source_format}"

    reader = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", source_format)
        .option("cloudFiles.schemaLocation", schema_location)
        # New columns appearing in later files (e.g. discount_pct) are added automatically
        # rather than breaking the stream.
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        # Columns that don't fit the inferred schema at all are captured, not dropped.
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")
        .option("recursiveFileLookup", "true")
    )
    if source_format == "csv":
        reader = reader.option("header", "true")

    df = (
        reader.load(source_path)
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_source_format", F.lit(source_format))
        .withColumn("_ingest_ts", F.current_timestamp())
    )

    query = (
        df.writeStream.format("delta")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema", "true")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
    )
    query.awaitTermination()
    return query.lastProgress

# COMMAND ----------

# MAGIC %md
# MAGIC Note: `amount` is inferred inconsistently across CSV (string) and JSON (double) by
# MAGIC Auto Loader's schema inference in some edge cases, and CSV has no native "null" type -
# MAGIC that's fine and expected in Bronze: **Bronze stores things as close to the raw source as
# MAGIC possible**; type-casting and validation is a Silver-layer responsibility.

# COMMAND ----------

csv_progress = ingest_source("csv", LANDING_CSV)
print("CSV batch progress:", csv_progress)

# COMMAND ----------

json_progress = ingest_source("json", LANDING_JSON)
print("JSON batch progress:", json_progress)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Sanity checks

# COMMAND ----------

display(spark.sql(f"""
    SELECT _source_format, count(*) AS row_count,
           min(_ingest_ts) AS first_ingested, max(_ingest_ts) AS last_ingested
    FROM {BRONZE_TABLE}
    GROUP BY _source_format
"""))

# COMMAND ----------

display(spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}"))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Basic performance housekeeping
# MAGIC Small-file compaction. On a real schedule this would run periodically (e.g. nightly),
# MAGIC not after every micro-batch.

# COMMAND ----------

spark.sql(f"OPTIMIZE {BRONZE_TABLE}")


In [0]:
%sql
select * from bronze_transactions